In [13]:
# [CELL 1] STUDENT CONFIGURATION

from pathlib import Path
import csv
import json
import shutil
from collections import Counter

LAST_NAME = "Bautista"
STUDENT_ID = "TUPM-26-3506"

SEED_DIGIT = int(STUDENT_ID[-1])

TARGET_YEAR = 2015 + SEED_DIGIT

print("Target Year:", TARGET_YEAR)

Target Year: 2021


In [14]:
# [CELL 2] DATASET PATH

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

dataset_path = (
    DATA_DIR /
    "Global_Cybersecurity_Threats_2015-2024.csv"
)

filtered_csv = OUTPUT_DIR / (
    f"{LAST_NAME}_{STUDENT_ID}_cyber_filtered.csv"
)

summary_json = OUTPUT_DIR / (
    f"{LAST_NAME}_{STUDENT_ID}_cyber_summary.json"
)

backup_json = BACKUP_DIR / summary_json.name

report_file = OUTPUT_DIR / (
    f"{LAST_NAME}_{STUDENT_ID}_cyber_report.txt"
)

In [15]:
# [CELL 3] DATASET PROCESSING

selected_records = []

attack_counter = Counter()

total_records = 0

total_loss = 0

with open(
    dataset_path,
    "r",
    encoding="utf-8"
) as file:

    reader = csv.DictReader(file)

    for index, row in enumerate(reader):

        total_records += 1

        try:

            year = int(row["Year"])

        except ValueError:
            continue

        if year != TARGET_YEAR:
            continue

        try:

            loss = float(
                row["Financial Loss (in Million $)"]
            )

        except ValueError:
            continue

        attack_type = row["Attack Type"]

        attack_counter[attack_type] += 1

        total_loss += loss

        row["_loss"] = loss
        row["_index"] = index

        selected_records.append(row)

print("Total records:", total_records)
print(
    "Target year records:",
    len(selected_records)
)

Total records: 3000
Target year records: 299


In [16]:
# [CELL 4] LOSSES

incident_count = len(selected_records)

if incident_count:

    average_loss = (
        total_loss / incident_count
    )

else:

    average_loss = 0

print(
    "Total Loss:",
    round(total_loss,2)
)

print(
    "Average Loss:",
    round(average_loss,2)
)


Total Loss: 15873.41
Average Loss: 53.09


In [17]:
# [CELL 5] TOP 5 COSTLIEST INCIDENTS 

top_five = sorted(

    selected_records,

    key=lambda x: (
        -x["_loss"],
        x["_index"]
    )

)[:5]

print("Five Costliest Incidents")

for item in top_five:

    print(
        item["Attack Type"],
        item["_loss"]
    )

Five Costliest Incidents
DDoS 99.81
Ransomware 99.77
SQL Injection 98.76
SQL Injection 98.57
SQL Injection 98.09


In [18]:
# [CELL 6] EXPORTING 

if selected_records:

    fields = [
        k for k in selected_records[0]
        if not k.startswith("_")
    ]

    with open(
        filtered_csv,
        "w",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=fields
        )

        writer.writeheader()

        for row in selected_records:

            clean_row = {
                k:v
                for k,v in row.items()
                if not k.startswith("_")
            }

            writer.writerow(clean_row)

print("Created:", filtered_csv)


Created: output\Bautista_TUPM-26-3506_cyber_filtered.csv


In [19]:
# [CELL 7] SUMMARY JSON

summary = {

    "target_year":
    TARGET_YEAR,

    "total_records":
    total_records,

    "incidents_in_year":
    incident_count,

    "count_per_attack_type":
    dict(attack_counter),

    "total_financial_loss":
    round(total_loss,2),

    "average_financial_loss":
    round(average_loss,2),

    "five_costliest_incidents":[

        {
            "attack_type":
            i["Attack Type"],

            "loss":
            i["_loss"]
        }

        for i in top_five
    ]
}

with open(
    summary_json,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        summary,
        file,
        indent=4
    )

print("Created:", summary_json)


Created: output\Bautista_TUPM-26-3506_cyber_summary.json


In [20]:
# [CELL 8] BACKUP

shutil.copy2(
    summary_json,
    backup_json
)

print(
    "Backup Exists:",
    backup_json.exists()
)

Backup Exists: True


In [21]:
# [CELL 9] REPORT FILE

with open(
    report_file,
    "w",
    encoding="utf-8"
) as report:

    report.write(
        "CYBERSECURITY INCIDENT REPORT\n"
    )

    report.write(
        "-" * 40 + "\n"
    )

    report.write(
        f"Target Year: {TARGET_YEAR}\n"
    )

    report.write(
        f"Total Dataset Records: "
        f"{total_records}\n"
    )

    report.write(
        f"Incidents In Year: "
        f"{incident_count}\n"
    )

    report.write(
        f"Total Loss: "
        f"{total_loss:.2f}\n"
    )

    report.write(
        f"Average Loss: "
        f"{average_loss:.2f}\n"
    )

    report.write(
        "\nCOUNT PER ATTACK TYPE\n"
    )

    for attack, count in attack_counter.items():

        report.write(
            f"{attack}: {count}\n"
        )

print("Report Created:", report_file)

Report Created: output\Bautista_TUPM-26-3506_cyber_report.txt


In [22]:
# [CELL 10] ASSESSMENT OUTPUT

print("\nCYBERSECURITY INCIDENT REPORT")
print("-" * 40)

print("Target Year:", TARGET_YEAR)

print(
    "Total Records:",
    total_records
)

print(
    "Incidents In Year:",
    incident_count
)

print(
    "Count Per Attack Type:"
)

for attack, count in attack_counter.items():

    print(
        attack,
        ":",
        count
    )

print(
    "Total Loss:",
    round(total_loss,2)
)

print(
    "Average Loss:",
    round(average_loss,2)
)

print(
    "Filtered CSV:",
    filtered_csv.name
)

print(
    "JSON Summary:",
    summary_json.name
)

print(
    "Backup Verification:",
    backup_json.exists()
)

print(
    "Report File:",
    report_file.name
)


CYBERSECURITY INCIDENT REPORT
----------------------------------------
Target Year: 2021
Total Records: 3000
Incidents In Year: 299
Count Per Attack Type:
Phishing : 41
SQL Injection : 59
Malware : 49
DDoS : 47
Ransomware : 55
Man-in-the-Middle : 48
Total Loss: 15873.41
Average Loss: 53.09
Filtered CSV: Bautista_TUPM-26-3506_cyber_filtered.csv
JSON Summary: Bautista_TUPM-26-3506_cyber_summary.json
Backup Verification: True
Report File: Bautista_TUPM-26-3506_cyber_report.txt
